# 🔬 Real-World Stress Test Series — Episode 03
## Operations & Supply Chain: Can observed sales hide unmet demand?

**Research question:** When sales fall to zero during a simulated stockout, would a simple forecasting baseline interpret the missing sales as lower demand?

### Important design choice
The Olist public e-commerce dataset does **not** contain reliable item-level inventory availability or true lost-demand labels. Therefore, this notebook uses Olist to build an **order-activity proxy**, then creates a clearly labelled **synthetic stockout scenario**. It does not claim to recover true demand or prove what customers would have purchased.

The analysis is a focused stress-test experiment, not a production demand-forecasting system.

## 1. What this notebook will do

1. Load the Olist orders and order-items tables from public raw CSV URLs.
2. Check data quality and identify delivered orders.
3. Aggregate delivered order items into daily observed item-level sales for a manageable set of frequently sold products.
4. Create a daily time series with zero-sales days filled in.
5. Train a simple moving-average baseline using pre-test observations only.
6. Simulate a 14-day stockout by masking observed sales as zero for one selected product during a test period.
7. Compare baseline forecasts with observed sales and show how zeros can distort a naïve interpretation.
8. Save figures and summary tables for the LinkedIn post.

**No model accuracy claim will be made for the simulated stockout period.** The true lost demand during a stockout is unknown.

## 2. Dataset and source

Dataset: **Brazilian E-Commerce Public Dataset by Olist** (Kaggle).

The notebook downloads the `orders` and `order_items` CSV files from a public mirror so it can run without a Kaggle API token. Public mirrors can change or become unavailable; if a URL fails, download the dataset from Kaggle and update the URLs in the next cell.

Source page: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

The source dataset is third-party data. Follow the dataset publisher's terms and attribution requirements.

In [ ]:
# If needed, install dependencies in your environment:
# %pip install pandas numpy matplotlib requests

from pathlib import Path
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
FIG_DIR = OUTPUT_DIR / "figures"
RESULTS_DIR = OUTPUT_DIR / "results"

for folder in [DATA_DIR, FIG_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Folders ready:", BASE_DIR.resolve())

## 3. Download data

This cell uses raw CSV URLs from a public GitHub mirror. If download fails, manually download the two files from the Kaggle dataset page and place them in `data/` using the exact filenames shown below.

In [ ]:
DATA_URLS = {
    "orders": "https://raw.githubusercontent.com/olistbr/brazilian-ecommerce/master/olist_orders_dataset.csv",
    "order_items": "https://raw.githubusercontent.com/olistbr/brazilian-ecommerce/master/olist_order_items_dataset.csv",
}

def download_if_missing(name, url):
    path = DATA_DIR / f"olist_{name}_dataset.csv"
    if path.exists():
        print(f"Using existing file: {path}")
        return path
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    path.write_bytes(response.content)
    print(f"Downloaded: {path} ({path.stat().st_size:,} bytes)")
    return path

try:
    orders_path = download_if_missing("orders", DATA_URLS["orders"])
    items_path = download_if_missing("order_items", DATA_URLS["order_items"])
except Exception as exc:
    raise RuntimeError(
        "Automatic download failed. Download the Olist dataset from "
        "https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce and place "
        "olist_orders_dataset.csv and olist_order_items_dataset.csv in the data/ folder."
    ) from exc

## 4. Load and inspect the tables

We use the order status and purchase timestamp from the orders table, and product ID and price from order items. Product IDs are identifiers, not meaningful numerical predictors.

In [ ]:
orders = pd.read_csv(orders_path, parse_dates=["order_purchase_timestamp"])
items = pd.read_csv(items_path)

print("Orders shape:", orders.shape)
print("Order items shape:", items.shape)
display(orders.head())
display(items.head())

print("\\nOrder status counts:")
display(orders["order_status"].value_counts(dropna=False).to_frame("count"))

## 5. Data quality checks and define observed sales

We count **delivered order items** as an observed-sales proxy. This choice reduces ambiguity from cancelled orders, but it is not a perfect measure of when the customer actually received or consumed a product.

We aggregate by purchase date and product ID. This is an order-activity series, not a complete retail demand dataset.

In [ ]:
required_orders = {"order_id", "order_status", "order_purchase_timestamp"}
required_items = {"order_id", "product_id", "order_item_id", "price"}
assert required_orders.issubset(orders.columns), f"Missing columns: {required_orders - set(orders.columns)}"
assert required_items.issubset(items.columns), f"Missing columns: {required_items - set(items.columns)}"

print("Missing order IDs in orders:", orders["order_id"].isna().sum())
print("Missing product IDs in items:", items["product_id"].isna().sum())
print("Duplicate order IDs in orders:", orders["order_id"].duplicated().sum())

delivered_orders = orders.loc[
    orders["order_status"].eq("delivered"),
    ["order_id", "order_purchase_timestamp"]
].dropna()

delivered_items = items.merge(delivered_orders, on="order_id", how="inner")
delivered_items["purchase_date"] = delivered_items["order_purchase_timestamp"].dt.floor("D")

print("Delivered orders:", delivered_orders["order_id"].nunique())
print("Delivered order-item rows:", len(delivered_items))
print("Unique products in delivered items:", delivered_items["product_id"].nunique())
display(delivered_items[["order_id", "product_id", "purchase_date", "price"]].head())

## 6. Select a product and create a daily series

To keep this notebook runnable and the plots readable, we choose one of the most frequently ordered products among delivered orders. This is a convenience selection, not a claim that it represents the whole Brazilian market.

We fill calendar dates with zero observed order-item counts. A zero means no delivered order-item activity was recorded for that product on that date; it does **not** prove there was no demand.

In [ ]:
product_item_counts = delivered_items.groupby("product_id").size().sort_values(ascending=False)
if product_item_counts.empty:
    raise ValueError("No delivered order items available after filtering.")

SELECTED_PRODUCT_ID = product_item_counts.index[0]
selected = delivered_items.loc[delivered_items["product_id"].eq(SELECTED_PRODUCT_ID)].copy()

date_index = pd.date_range(
    delivered_items["purchase_date"].min(),
    delivered_items["purchase_date"].max(),
    freq="D"
)
daily_sales = (
    selected.groupby("purchase_date")["order_item_id"]
    .count()
    .reindex(date_index, fill_value=0)
    .rename("observed_order_items")
    .to_frame()
)
daily_sales.index.name = "date"

print("Selected product:", SELECTED_PRODUCT_ID)
print("Observed order-item rows:", len(selected))
print("Daily series dates:", len(daily_sales))
display(daily_sales.head())

## 7. Visualize the historical observed-sales proxy

This chart shows recorded daily order-item activity for the selected product. The series can be intermittent and seasonal. Remember: observed orders are not the same as latent customer demand.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(daily_sales.index, daily_sales["observed_order_items"], linewidth=1)
ax.set_title("Olist: Daily observed order-item activity for one product")
ax.set_xlabel("Purchase date")
ax.set_ylabel("Delivered order items recorded")
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "historical_observed_sales.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Define a chronological train/test split

We use an early period for fitting the simple baseline and reserve the later period for the stress experiment. This avoids randomly mixing future and past observations.

The baseline is a simple trailing 7-day moving average. It is intentionally transparent; this is a demonstration of the measurement problem, not a competition to build the best forecaster.

In [ ]:
split_idx = int(len(daily_sales) * 0.8)
if split_idx < 14 or len(daily_sales) - split_idx < 21:
    raise ValueError("Not enough daily observations for this split and a 14-day stress window.")

train = daily_sales.iloc[:split_idx].copy()
test = daily_sales.iloc[split_idx:].copy()

print("Train period:", train.index.min().date(), "to", train.index.max().date(), "| days:", len(train))
print("Test period:", test.index.min().date(), "to", test.index.max().date(), "| days:", len(test))

## 9. Build a simple baseline forecast

The forecast for each test day is the mean of the previous 7 days of **observed** sales. After a simulated stockout, a naïve rolling forecast may be pulled down by the artificial zeros. This is the mechanism we will examine.

For a real inventory system, the model should distinguish unavailable inventory from genuine zero demand.

In [ ]:
full_observed = daily_sales["observed_order_items"].astype(float)
baseline_forecast = pd.Series(index=test.index, dtype=float)

history = train["observed_order_items"].astype(float).tolist()
for date in test.index:
    baseline_forecast.loc[date] = float(np.mean(history[-7:])) if history else 0.0
    history.append(float(daily_sales.loc[date, "observed_order_items"]))

baseline_forecast = baseline_forecast.clip(lower=0)
display(pd.DataFrame({
    "observed_order_items": test["observed_order_items"],
    "baseline_7day_mean": baseline_forecast
}).head(10))

## 10. Simulate a 14-day stockout

We create an artificial stockout window in the test period and mask the selected product's observed sales as zero. The stockout window is chosen away from the beginning/end of the test set.

**Critical limitation:** Olist does not provide inventory availability for this product/date series. The stockout dates are simulated and must never be described as a real stockout event.

In [ ]:
# Choose a 14-day window near the middle of the held-out test period.
window_length = 14
start_pos = max(3, (len(test) // 2) - (window_length // 2))
if start_pos + window_length >= len(test):
    start_pos = len(test) - window_length - 1
if start_pos < 0:
    raise ValueError("Test period is too short for a 14-day stockout window.")

stockout_dates = test.index[start_pos:start_pos + window_length]
stress_observed = test["observed_order_items"].astype(float).copy()
stress_observed.loc[stockout_dates] = 0.0

print("Simulated stockout:", stockout_dates.min().date(), "to", stockout_dates.max().date())
print("Number of simulated days:", len(stockout_dates))
print("Original observed item count during window:", int(test.loc[stockout_dates, "observed_order_items"].sum()))
print("Masked count during window:", int(stress_observed.loc[stockout_dates].sum()))

## 11. Recompute the rolling forecast after masked stockout observations

This compares two scenarios:
- **Reference:** forecast updated using the original observed sales proxy.
- **Simulated stockout:** forecast updated using zero-masked sales during the 14-day window.

The comparison isolates how the artificial zeros affect a simple rolling baseline. It does not establish the true lost demand during the stockout.

In [ ]:
def rolling_one_step_forecast(train_series, test_series):
    history = train_series.astype(float).tolist()
    preds = []
    for value in test_series.astype(float).tolist():
        preds.append(float(np.mean(history[-7:])) if history else 0.0)
        history.append(value)
    return pd.Series(preds, index=test_series.index, dtype=float)

reference_forecast = rolling_one_step_forecast(train["observed_order_items"], test["observed_order_items"])
stockout_forecast = rolling_one_step_forecast(train["observed_order_items"], stress_observed)

comparison = pd.DataFrame({
    "original_observed_sales": test["observed_order_items"].astype(float),
    "simulated_stockout_observed_sales": stress_observed,
    "reference_forecast": reference_forecast,
    "forecast_after_stockout_masking": stockout_forecast,
})
comparison["forecast_difference"] = comparison["forecast_after_stockout_masking"] - comparison["reference_forecast"]
display(comparison.loc[stockout_dates].head(14))

## 12. Figure 1 — Observed sales and simulated stockout

This is the primary LinkedIn visual. It shows the 14-day period in which recorded sales are artificially set to zero.

Do not label the zeros as true demand. Label them as **simulated observed sales during a stockout**.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test.index, test["observed_order_items"], label="Original observed sales proxy", linewidth=1.5)
ax.plot(test.index, stress_observed, label="After simulated stockout masking", linewidth=1.5, linestyle="--")
ax.axvspan(stockout_dates.min(), stockout_dates.max(), alpha=0.18, label="Simulated 14-day stockout")
ax.set_title("Simulated stockout: observed sales are masked to zero")
ax.set_xlabel("Date")
ax.set_ylabel("Observed order-item count")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "simulated_stockout_observed_sales.png", dpi=180, bbox_inches="tight")
plt.show()

## 13. Figure 2 — Forecast effect of artificial zeros

This figure compares the simple rolling forecast when it sees the original observations versus when the simulated stockout zeros are fed back into the forecast process.

The important point is not that the forecast reveals true demand. It is that **the input data-collection process changes the forecast**.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test.index, reference_forecast, label="Reference rolling forecast", linewidth=1.8)
ax.plot(test.index, stockout_forecast, label="Forecast after simulated stockout zeros", linewidth=1.8, linestyle="--")
ax.axvspan(stockout_dates.min(), stockout_dates.max(), alpha=0.18, label="Simulated stockout window")
ax.set_title("How simulated stockout zeros affect a 7-day moving-average forecast")
ax.set_xlabel("Date")
ax.set_ylabel("Forecast order items per day")
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(FIG_DIR / "forecast_effect_of_stockout.png", dpi=180, bbox_inches="tight")
plt.show()

## 14. Quantify the forecast change

We report the average forecast during the stockout window and in the 14 days after it. These are descriptive comparisons between two simulated input streams, not estimates of real lost sales.

A 14-day stockout is only one scenario; results depend on the selected product and window.

In [ ]:
post_window = test.index[test.index > stockout_dates.max()][:14]
summary_rows = [
    {
        "period": "Stockout window",
        "start_date": stockout_dates.min().date().isoformat(),
        "end_date": stockout_dates.max().date().isoformat(),
        "reference_forecast_mean": reference_forecast.loc[stockout_dates].mean(),
        "stockout_scenario_forecast_mean": stockout_forecast.loc[stockout_dates].mean(),
        "mean_forecast_difference": comparison.loc[stockout_dates, "forecast_difference"].mean(),
    }
]
if len(post_window) > 0:
    summary_rows.append({
        "period": "First days after stockout (up to 14)",
        "start_date": post_window.min().date().isoformat(),
        "end_date": post_window.max().date().isoformat(),
        "reference_forecast_mean": reference_forecast.loc[post_window].mean(),
        "stockout_scenario_forecast_mean": stockout_forecast.loc[post_window].mean(),
        "mean_forecast_difference": comparison.loc[post_window, "forecast_difference"].mean(),
    })

summary = pd.DataFrame(summary_rows)
display(summary.round(3))
summary.to_csv(RESULTS_DIR / "stockout_stress_summary.csv", index=False)
comparison.to_csv(RESULTS_DIR / "daily_forecast_comparison.csv", index_label="date")
print("Saved summary and daily comparison tables.")

## 15. Optional sensitivity check: repeat the scenario for another popular product

One product and one date window are not enough to generalize. This optional cell checks a few of the most frequently ordered products with the same chronological split and middle-of-test stockout design. It is still a simulated experiment, not proof of general market behaviour.

In [ ]:
def build_daily_series_for_product(product_id):
    prod = delivered_items.loc[delivered_items["product_id"].eq(product_id)]
    series = prod.groupby("purchase_date")["order_item_id"].count().reindex(date_index, fill_value=0)
    series.index.name = "date"
    return series.astype(float)

sensitivity_rows = []
for product_id in product_item_counts.head(5).index:
    s = build_daily_series_for_product(product_id)
    tr, te = s.iloc[:split_idx], s.iloc[split_idx:]
    if len(te) <= window_length + 5:
        continue
    pos = max(3, (len(te) // 2) - (window_length // 2))
    if pos + window_length >= len(te):
        pos = len(te) - window_length - 1
    if pos < 0:
        continue
    dates = te.index[pos:pos + window_length]
    masked = te.copy()
    masked.loc[dates] = 0.0
    ref_pred = rolling_one_step_forecast(tr, te)
    stress_pred = rolling_one_step_forecast(tr, masked)
    post_dates = te.index[te.index > dates.max()][:14]
    sensitivity_rows.append({
        "product_id": product_id,
        "stockout_start": dates.min().date().isoformat(),
        "stockout_end": dates.max().date().isoformat(),
        "reference_forecast_during_stockout": ref_pred.loc[dates].mean(),
        "stress_forecast_during_stockout": stress_pred.loc[dates].mean(),
        "reference_forecast_after_stockout": ref_pred.loc[post_dates].mean() if len(post_dates) else np.nan,
        "stress_forecast_after_stockout": stress_pred.loc[post_dates].mean() if len(post_dates) else np.nan,
    })

sensitivity = pd.DataFrame(sensitivity_rows)
display(sensitivity.round(3))
sensitivity.to_csv(RESULTS_DIR / "product_sensitivity_check.csv", index=False)

## 16. Interpretation — what can and cannot be concluded

### Supported conclusion
In this controlled simulation, masking observed sales to zero changes the input history seen by a rolling-average forecasting baseline. That can change subsequent forecasts.

### Not supported by this dataset alone
- That a real Olist product was out of stock on these dates.
- That zero observed orders means zero customer demand.
- The number of customers who wanted to buy but could not.
- The true demand that would have occurred during the simulated stockout.
- That this baseline is a good production forecasting model.

### Practical business lesson
Inventory systems should record stock availability and distinguish **zero sales because there was no demand** from **zero sales because the item was unavailable**. Otherwise, the model may learn from censored sales data as though it were complete demand.

## 17. Figures and files generated

The notebook saves:

- `outputs/figures/historical_observed_sales.png`
- `outputs/figures/simulated_stockout_observed_sales.png`
- `outputs/figures/forecast_effect_of_stockout.png`
- `outputs/results/stockout_stress_summary.csv`
- `outputs/results/daily_forecast_comparison.csv`
- `outputs/results/product_sensitivity_check.csv`

For LinkedIn, use the **simulated stockout chart** as the main figure and the **forecast effect chart** as the optional second figure. Use actual outputs from this notebook only; do not invent numerical findings.

## 18. Suggested LinkedIn framing (fill in only after running)

**🔬 REAL WORLD STRESS TEST SERIES — EPISODE 03 / OPERATIONS & SUPPLY CHAIN**

Can a sales forecasting baseline mistake a stockout for falling demand?

Using Olist order data, I built a daily observed-sales proxy for a frequently ordered product. I then simulated a 14-day stockout by masking recorded sales to zero and compared how a simple 7-day moving-average forecast responded.

**What I found:** [Insert the exact change from `stockout_stress_summary.csv` after running the notebook.]

The important limitation: Olist does not provide item-level inventory availability or true lost-demand labels for this experiment. This is a simulated stress test, not evidence of a real stockout or a measurement of lost sales.

The takeaway: observed sales are not always the same as customer demand. Without inventory-availability signals, zero sales can be ambiguous.

#DataScience #SupplyChain #OperationsAnalytics #Forecasting #Python

**Before publishing:** verify the actual outputs, describe the selected product/window as a simulation, and avoid claiming the model under-forecasted true demand unless you have ground-truth demand data.

## 19. Reproducibility and attribution

- Dataset source: Olist Brazilian E-Commerce Public Dataset, available through Kaggle.
- The raw data is not redistributed in this notebook; it is downloaded to the local `data/` folder for execution.
- This is an educational stress-test experiment. It is not a production inventory or demand-planning model.
- Public availability of this notebook does not change the rights or terms attached to third-party data.